# Maintained experiment notebook

Original executed notebook and historical outputs: `../archive/original-notebooks/`.
Run this copy in a fresh kernel; outputs are intentionally cleared. Full runs download CIFAR-100 and, when enabled, pretrained weights.
Read `../docs/REPRODUCIBILITY.md` for maintenance changes and evaluation limits.


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

batch_size = 128
num_epochs = 20
learning_rate = 1e-4
weight_decay = 1e-4


transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.Resize(224),  
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

transform_test = transforms.Compose([
    transforms.Resize(224),  
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

In [ ]:
train_dataset = datasets.CIFAR100(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform_train
)

test_dataset = datasets.CIFAR100(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform_test
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=4
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=4
)

In [ ]:
model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)


num_classes = 100
in_features = model.heads.head.in_features
model.heads.head = nn.Linear(in_features, num_classes)


model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)


scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)


In [ ]:
train_losses = []
test_losses = []
train_accs = []
test_accs = []


best_model_path = 'best_vit_cifar100.pth'
best_acc = 0.0


def train(epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
    for batch_idx, (inputs, targets) in enumerate(progress_bar):
        inputs, targets = inputs.to(device), targets.to(device)
        

        optimizer.zero_grad()
        

        outputs = model(inputs)
        loss = criterion(outputs, targets)
        

        loss.backward()
        optimizer.step()
        

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        

        progress_bar.set_postfix({
            'loss': running_loss/(batch_idx+1), 
            'acc': 100.*correct/total
        })
    
    train_loss = running_loss/len(train_loader)
    train_acc = 100.*correct/total
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    return train_loss, train_acc


def test(epoch):
    global best_acc
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        progress_bar = tqdm(test_loader, desc='Testing')
        for batch_idx, (inputs, targets) in enumerate(progress_bar):
            inputs, targets = inputs.to(device), targets.to(device)
            

            outputs = model(inputs)
            loss = criterion(outputs, targets)
            

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            

            progress_bar.set_postfix({
                'loss': running_loss/(batch_idx+1), 
                'acc': 100.*correct/total
            })
    
    test_loss = running_loss/len(test_loader)
    test_acc = 100.*correct/total
    
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    

    if test_acc > best_acc:
        print(f'Saving best model with accuracy: {test_acc:.2f}%')
        best_acc = test_acc
        torch.save(model.state_dict(), best_model_path)
    
    return test_loss, test_acc


for epoch in range(num_epochs):
    train_loss, train_acc = train(epoch)
    test_loss, test_acc = test(epoch)
    

    scheduler.step()
    
    print(f'Epoch: {epoch+1}/{num_epochs}')
    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
    print(f'Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%')
    print('-' * 70)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss Curves')

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Accuracy')
plt.plot(test_accs, label='Test Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.title('Accuracy Curves')

plt.tight_layout()
plt.savefig('vit_cifar100_training_curves.png')
plt.show()

model.load_state_dict(torch.load(best_model_path))
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for inputs, targets in tqdm(test_loader, desc='Final Evaluation'):
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

print(f'Best model accuracy on test set: {100.*correct/total:.2f}%')